<a href="https://colab.research.google.com/github/Kantheephob/CS372_Reinforcement_Learning/blob/main/21_game_re7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# ====== INSTALL ======
try:
    import ipywidgets
except:
    !pip install ipywidgets

# ====== IMPORT ======
import numpy as np
import random
import ipywidgets as widgets
from IPython.display import display
import time

# ====== SETTINGS ======
alpha = 0.1
gamma = 0.9
epsilon = 0.1
pretrain_episodes = 1000000

# ====== Q-TABLE ======
Q = {}

def get_q(state):
    if state not in Q:
        Q[state] = np.zeros(2)
    return Q[state]

# ====== GAME UTILS ======
def new_deck():
    deck = [i for i in range(1, 12)]
    random.shuffle(deck)
    return deck

def get_visible_sum(cards):
    if len(cards) <= 1: return 0
    return sum(cards[1:])

# ====== PRETRAIN ======
print(f"กำลัง Pretrain AI ระบบไพ่ 11 ใบ ({pretrain_episodes} รอบ)... ")
for ep in range(pretrain_episodes):
    deck_pre = new_deck()
    ai_c = [deck_pre.pop(), deck_pre.pop()]
    pl_c = [deck_pre.pop(), deck_pre.pop()]
    ai_s, pl_s = False, False
    ai_h = []
    while True:
        if not pl_s:
            if sum(pl_c) < 13:
                if len(deck_pre) > 0:
                    pl_c.append(deck_pre.pop())
                    if sum(pl_c) > 21:
                        for s, a in ai_h: get_q(s)[a] += alpha * (1 - get_q(s)[a])
                        break
                else: pl_s = True
            else: pl_s = True
        if not ai_s:
            st = (sum(ai_c), get_visible_sum(pl_c))
            act = random.choice([0, 1]) if random.random() < epsilon else np.argmax(get_q(st))
            ai_h.append((st, act))
            if act == 1 and len(deck_pre) > 0:
                ai_c.append(deck_pre.pop())
                if sum(ai_c) > 21:
                    for s, a in ai_h: get_q(s)[a] += alpha * (-1 - get_q(s)[a])
                    break
            else: ai_s = True
        if (pl_s and ai_s) or len(deck_pre) == 0:
            rew = 1 if sum(ai_c) > sum(pl_c) else (-1 if sum(ai_c) < sum(pl_c) else 0)
            for s, a in ai_h: get_q(s)[a] += alpha * (rew - get_q(s)[a])
            break

print("Pretrain เสร็จสิ้น!")

# ====== UI & STATE ======
output_main = widgets.Output()
output_log = widgets.Output()
btn_hit = widgets.Button(description="Hit (จั่ว)", button_style='info')
btn_stay = widgets.Button(description="Stay (หยุด)", button_style='warning')
train_toggle = widgets.ToggleButton(value=False, description='Online Train: OFF', button_style='danger')

def on_train_toggle(change):
    if change['new']:
        train_toggle.description = 'Online Train: ON'
        train_toggle.button_style = 'success'
    else:
        train_toggle.description = 'Online Train: OFF'
        train_toggle.button_style = 'danger'
train_toggle.observe(on_train_toggle, 'value')

current_deck = []
player_cards = []
ai_cards = []
player_stayed = False
ai_stayed = False
game_active = False
log_history = []
ai_round_history = []

def update_ui(show_all=False):
    with output_main:
        output_main.clear_output(wait=True)
        print("="*35)
        print(f"Deck 1-11 (No Repeats) | Remaining: {len(current_deck)}")
        print("-"*35)
        print(f"Player: {player_cards} (Total: {sum(player_cards)})")
        if show_all: print(f"AI:     {ai_cards} (Total: {sum(ai_cards)})")
        else: print(f"AI:     {['?'] + ai_cards[1:]}")
        print("="*35)
    with output_log:
        output_log.clear_output(wait=True)
        print("--- Action Log ---")
        for l in log_history[-10:]: print(l)

def new_round():
    global current_deck, player_cards, ai_cards, player_stayed, ai_stayed, game_active, ai_round_history, log_history
    current_deck = new_deck()
    player_cards = [current_deck.pop(), current_deck.pop()]
    ai_cards = [current_deck.pop(), current_deck.pop()]
    player_stayed, ai_stayed, game_active = False, False, True
    ai_round_history, log_history = [], ["--- เริ่มเกมใหม่ ---"]
    btn_hit.disabled, btn_stay.disabled = False, False
    update_ui()

def end_game(winner, reason):
    global game_active
    game_active = False
    btn_hit.disabled, btn_stay.disabled = True, True
    reward = 1 if winner == "AI" else (-1 if winner == "Player" else 0)
    log_history.append(f"-> {reason} | {winner} Wins!")

    # ONLINE TRAINING LOGIC
    if train_toggle.value:
        for state, action in ai_round_history:
            current_q = get_q(state)[action]
            # TD-Update style
            get_q(state)[action] += alpha * (reward - current_q)
        log_history.append("[System] AI learned from this round.")

    update_ui(show_all=True)
    time.sleep(3)
    new_round()

def ai_turn():
    global ai_stayed, player_stayed
    if not game_active: return

    state = (sum(ai_cards), get_visible_sum(player_cards))
    # If Online Train is ON, use Epsilon for exploration
    if train_toggle.value and random.random() < epsilon:
        action = random.choice([0, 1])
    else:
        action = np.argmax(get_q(state))

    if len(current_deck) == 0: action = 0

    if action == 1:
        ai_round_history.append((state, action))
        card = current_deck.pop()
        ai_cards.append(card)
        log_history.append(f"AI hit -> ได้ไพ่ {card}")
        ai_stayed = False
        if sum(ai_cards) > 21:
            end_game("Player", "AI Bust")
            return
        player_stayed = False
        log_history.append("ตากลับมาที่ Player...")
    else:
        ai_round_history.append((state, action))
        ai_stayed = True
        log_history.append("AI stays.")
        if player_stayed:
            check_win()
            return

    update_ui()

def check_win():
    p, a = sum(player_cards), sum(ai_cards)
    if p > 21: end_game("AI", "Player Bust")
    elif a > 21: end_game("Player", "AI Bust")
    elif p > a: end_game("Player", "Higher Score")
    elif a > p: end_game("AI", "Higher Score")
    else: end_game("Draw", "Push")

def on_hit(b):
    if not game_active: return
    if len(current_deck) > 0:
        card = current_deck.pop()
        player_cards.append(card)
        log_history.append(f"Player hit -> ได้ไพ่ {card}")
        if sum(player_cards) > 21:
            end_game("AI", "Player Bust")
        else:
            ai_turn()
    else:
        log_history.append("ไพ่หมดกอง!")
        on_stay(None)

def on_stay(b):
    global player_stayed
    if not game_active: return
    player_stayed = True
    log_history.append("Player stays.")
    if ai_stayed: check_win()
    else: ai_turn()

btn_hit.on_click(on_hit)
btn_stay.on_click(on_stay)
display(widgets.HBox([btn_hit, btn_stay, train_toggle]), output_main, output_log)
new_round()

กำลัง Pretrain AI ระบบไพ่ 11 ใบ (1000000 รอบ)... 
Pretrain เสร็จสิ้น!


Output()

Output()